# Fase 2 - Limpieza y transformación de los datos

El objetivo principal de esta fase es preparar el conjunto de datos para su uso en modelos predictivos. En primer lugar, se lleva a cabo una limpieza que contempla la eliminación de variables irrelevantes o redundantes, así como la gestión de posibles valores atípicos que puedan afectar negativamente el rendimiento del modelo.

Cabe destacar que el conjunto de datos ya se encuentra razonablemente limpio, sin valores nulos ni registros duplicados, lo que simplifica considerablemente esta etapa y permite centrarse en transformaciones más específicas.

A continuación, se aplican las transformaciones necesarias —como la codificación de variables categóricas, el escalado de variables numéricas y la conversión de ordinales— para adecuar los datos al formato requerido por los algoritmos de clasificación que se emplearán en la siguiente fase: la selección y entrenamiento del modelo.

In [29]:
# Cargando las librerias necesarias
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Customizaciones
pd.set_option('display.max_columns', None)

In [30]:
# Cargando los datos de la fase anterior de EDA de un fichero pickle
df = pd.read_pickle('./data/df_eda.pkl')

# Verificar el dataframe
print(df.head())
print(df.info())
print(df.shape)

   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0  26.0     0.0     0.0   
1              0.0     1.0       1.0        1.0  26.0     1.0     1.0   
2              0.0     0.0       0.0        1.0  26.0     0.0     0.0   
3              0.0     1.0       1.0        1.0  28.0     1.0     0.0   
4              0.0     0.0       0.0        1.0  29.0     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     0.0      1.0                0.0   
1                   0.0           0.0     1.0      0.0                0.0   
2                   0.0           1.0     1.0      1.0                0.0   
3                   0.0           1.0     1.0      1.0                0.0   
4                   0.0           1.0     1.0      1.0                0.0   

   AnyHealthcare  NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex  \
0            1.0     

## Eliminar variables con poca variabilidad. Son variables binárias que tienen valores con más 95%, por lo tanto, no aportan valor predictivo al modelo.

En la fase anterior de EDA fue posible identificar dos variables que deben ser eliminadas de acuerdo con estes criterio. Las variables a seren eliminadas son:

CholCheck, HvyAlcoholConsump y AnyHealthcare



In [31]:
# Eliminar columnas con baja variabilidad del DataFrame
# df.drop(columns=['CholCheck', 'HvyAlcoholConsump', 'AnyHealthcare'], inplace=True)

# Verificar el dataframe después de eliminar las columnas
# print(df.head())
# print(df.shape)

## Tratamiento de los valores atípicos

Es necesario tratar los valores atípicos detectados en la fase anterior. Las variables con valores atípicos son: 

BMI, MentHlth y PhysHlth

Las, consideraciones, estrategias y justificativas para cada variable de cómo gestionar sus atípicos:

### BMI 
Consideración: Outliers suelen ser casos reales (obesidad extrema o bajo peso severo). Eliminarlos puede sesgar el modelo hacia pacientes “promedio”.
Estrategia: Capping o winsorización: limitar valores extremos al percentil 1 y 99 o al límite superior calculado con IQR.

### MentHlth y PhysHlth
Consideración: En ambos, no hay negativos; el límite inferior negativo en IQR es un artefacto matemático. Valores > 30 no existen (en BRFSS, 30 = todos los días).
Estrategia: Para ambas, no imputar ni eliminar; solo verificar y recodificar valores > 30 como 30 (si hubiera por error)

## BMI WINSORIZACIÓN

In [32]:
# BMI WINSORIZACIÓN (p1–p99)

q1, q99 = df['BMI'].quantile([0.01, 0.99])

print("=== ELIMINACIÓN DE ATÍPICOS EN BMI (p1–p99) ===")
print(f"Percentiles usados -> p1: {q1:.2f}, p99: {q99:.2f}")
print(f"Antes  -> min: {df['BMI'].min():.2f} | mediana: {df['BMI'].median():.2f} | max: {df['BMI'].max():.2f}")
print(f"Filas a eliminar -> por debajo de p1: {(df['BMI'] < q1).sum()} | por encima de p99: {(df['BMI'] > q99).sum()}")

# Filtrar df para mantener solo valores dentro del rango
df = df[(df['BMI'] >= q1) & (df['BMI'] <= q99)]

print(f"Después -> min: {df['BMI'].min():.2f} | mediana: {df['BMI'].median():.2f} | max: {df['BMI'].max():.2f}")
print(f"Tamaño del DataFrame después de filtrar BMI: {df.shape}\n")

=== ELIMINACIÓN DE ATÍPICOS EN BMI (p1–p99) ===
Percentiles usados -> p1: 19.00, p99: 52.00
Antes  -> min: 12.00 | mediana: 29.00 | max: 98.00
Filas a eliminar -> por debajo de p1: 649 | por encima de p99: 690
Después -> min: 19.00 | mediana: 29.00 | max: 52.00
Tamaño del DataFrame después de filtrar BMI: (67718, 22)



## MentHlth y PhysHlth

En las variables MentHlth y PhysHlth no se detectaron valores reales fuera de su rango válido (0–30 días). Los límites inferiores negativos observados durante el análisis corresponden únicamente a artefactos del cálculo estadístico mediante el método IQR y no representan datos inválidos. Por lo tanto, no se requiere aplicar ningún tratamiento adicional sobre estas variables.

## VERIFICACIÓN

In [33]:
# Verificación del tratamiento de los outliers
print("=== VERIFICACIÓN FINAL (describe) ===")
print(df[['BMI','MentHlth','PhysHlth']].describe().round(2))




=== VERIFICACIÓN FINAL (describe) ===
            BMI  MentHlth  PhysHlth
count  67718.00  67718.00  67718.00
mean      29.75      3.79      5.85
std        6.22      8.17     10.07
min       19.00      0.00      0.00
25%       25.00      0.00      0.00
50%       29.00      0.00      0.00
75%       33.00      2.00      6.00
max       52.00     30.00     30.00


Los valores atípicos fueron correctamente tratados porque:
    
    
    - Se eliminaron o ajustaron solo los valores incoherentes o extremos fuera de la realidad esperada.
    
    - Los valores que el IQR sigue señalando son extremos naturales del conjunto y deben permanecer para conservar la representatividad de los datos

## Transformaciones

Las transformaciones necesarias para el objetivo del proyecto es hacer un escalado en las variables numericas y categoricas ordinales.





In [34]:
# Variables a escalar

# Invertir escala de GenHlth antes de escalar por coherencia semántica
df['GenHlth'] = 6 - df['GenHlth']  # Convierte 1->5, 2->4, 3->3, 4->2, 5->1
cols_continuas = ['BMI', 'MentHlth', 'PhysHlth']
cols_ordinales = ['Age', 'Education', 'Income', 'GenHlth']
cols_a_escalar = cols_continuas + cols_ordinales

# Instanciar el escalador
scaler = StandardScaler()
# scaler = MinMaxScaler()

# Aplicar escalado y reemplazar en df
df[cols_a_escalar] = scaler.fit_transform(df[cols_a_escalar])

# Verificación
print("Resumen después del escalado:")
print(df[cols_a_escalar].describe().loc[['mean', 'std']].round(2))

Resumen después del escalado:
      BMI  MentHlth  PhysHlth  Age  Education  Income  GenHlth
mean -0.0       0.0       0.0  0.0       -0.0    -0.0      0.0
std   1.0       1.0       1.0  1.0        1.0     1.0      1.0


In [35]:
print(df.head())
print(df.shape)

   Diabetes_binary  HighBP  HighChol  CholCheck       BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0 -0.602885     0.0     0.0   
1              0.0     1.0       1.0        1.0 -0.602885     1.0     1.0   
2              0.0     0.0       0.0        1.0 -0.602885     0.0     0.0   
3              0.0     1.0       1.0        1.0 -0.281138     1.0     0.0   
4              0.0     0.0       0.0        1.0 -0.120264     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     0.0      1.0                0.0   
1                   0.0           0.0     1.0      0.0                0.0   
2                   0.0           1.0     1.0      1.0                0.0   
3                   0.0           1.0     1.0      1.0                0.0   
4                   0.0           1.0     1.0      1.0                0.0   

   AnyHealthcare  NoDocbcCost   GenHlth  MentHlth  PhysHlth  DiffWalk  Sex

In [36]:
# Cargar el DataFrame transformado a un fichero pickle

with open("./data/df_trans.pkl", "wb") as f:
    pickle.dump(df, f)

print("DataFrame guardado como df_trans.pkl")

DataFrame guardado como df_trans.pkl
